In [4]:
# Bad Channel Detection & Visualization

import mne
import numpy as np
from pathlib import Path
from xeeg_kit import (
    detect_bad_channels, 
    plot_bad_channels_3d, 
    print_anatomical_summary,
    get_bel_rename_map,
    BELStandardizer
)

In [5]:

# 1. Inputs (Using your specific Raw Data paths)
INPUT_FILE = "/mnt/movement/users/jaizor/xtra/data/eeg/MISP/MISP_P001_EC_Trial1.mff"
GPSC_FILE = "/mnt/movement/users/jaizor/xtra/data/eeg/ghw280_from_egig.gpsc" 

# 2. Load and Standardize Data
print(f"\n1. Loading {Path(INPUT_FILE).name}...")
raw = mne.io.read_raw(INPUT_FILE, preload=True, verbose=False)

# Apply renaming and montage
rename_map = get_bel_rename_map(raw)
if rename_map:
    raw.rename_channels(rename_map)

standardizer = BELStandardizer(gpsc_file=GPSC_FILE, rename_map=rename_map)
raw = standardizer.standardize(raw)
print("✅ BEL Montage applied successfully.")

# 3. Filtering (Important for accurate bad channel detection)
print("\n2. Applying filtering...")
raw_filt = raw.copy().filter(l_freq=1.0, h_freq=100.0, picks='eeg', verbose=False)
raw_filt.notch_filter(freqs=50.0, picks='eeg', method='spectrum_fit', verbose=False)
raw_filt._data = np.real(raw_filt._data).astype(np.float64)

# 4. Detect Bad Channels
bad_chs = detect_bad_channels(raw_filt, mad_threshold=20.0)
print(f"\n3. 🔍 Detected {len(bad_chs)} bad channels: {sorted(bad_chs)}")

# 5. Anatomical Summary & Visualization
if len(bad_chs) > 0:
    # Uses the bundled map by default to find regions
    print("\n4. Checking anatomical distribution...")
    print_anatomical_summary(bad_chs)
    
    # Plot in 3D
    print("\n5. Visualizing bad channels in 3D...")
    plot_bad_channels_3d(raw_filt, bad_chs)
else:
    print("No bad channels detected with current threshold.")


1. Loading MISP_P001_EC_Trial1.mff...
✅ BEL Montage applied successfully.

2. Applying filtering...

3. 🔍 Detected 12 bad channels: ['E139', 'E146', 'E186', 'E19', 'E268', 'E270', 'E52', 'E53', 'E8', 'E88', 'E94', 'REF CZ']

4. Checking anatomical distribution...

📍 Bad Channels by Anatomical Region:
region
Orbital/Eye             3
Lateral-Inferior/Jaw    3
Cerebellar/Neck         2
Inferior-Frontal/Jaw    2
Central                 1

5. Visualizing bad channels in 3D...

✅ Interactive 3D plot saved to: bad_channels_3d_labeled.html
